In [ ]:
# Google Colab setup
!pip install -q polars plotly gdown python-dotenv kailash-ml

# Mount Google Drive for datasets
from google.colab import drive
drive.mount("/content/drive")


# ════════════════════════════════════════════════════════════════════════
# ASCENT04 — Exercise 4: Anomaly Detection and Ensembles
# ════════════════════════════════════════════════════════════════════════
# OBJECTIVE: Dimensionality reduction on fraud data, anomaly detection
#   with multiple detectors, and ensemble scoring with EnsembleEngine.blend().
#
# TASKS:
#   1. Load fraud data and reduce dimensionality with UMAP
#   2. Compare UMAP vs t-SNE embeddings
#   3. Isolation Forest anomaly scoring
#   4. LOF and additional anomaly detectors
#   5. Ensemble anomaly scores with EnsembleEngine
#   6. Evaluate and visualise detection performance
# ════════════════════════════════════════════════════════════════════════


In [ ]:
# Copyright 2026 Terrene Foundation
# SPDX-License-Identifier: Apache-2.0
from __future__ import annotations

import numpy as np
import polars as pl
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
)
from sklearn.preprocessing import StandardScaler

from kailash_ml import ModelVisualizer, EnsembleEngine
from kailash_ml.interop import to_sklearn_input

from shared import ASCENTDataLoader

try:
    import umap
except ImportError:
    umap = None



### Data Loading


In [ ]:
loader = ASCENTDataLoader()
fraud = loader.load("ascent04", "credit_card_fraud.parquet")

print(f"=== Credit Card Fraud Data ===")
print(f"Shape: {fraud.shape}")
print(f"Fraud rate: {fraud['is_fraud'].mean():.4%}")

feature_cols = [c for c in fraud.columns if c not in ("is_fraud", "transaction_id")]

X, y, col_info = to_sklearn_input(
    fraud.drop_nulls(),
    feature_columns=feature_cols,
    target_column="is_fraud",
)

# TODO: Standardise X with StandardScaler
scaler = ____
X_scaled = ____



## TASK 1: Dimensionality reduction with UMAP


In [ ]:
# Sample for visualisation (UMAP on 284K rows is slow)
rng = np.random.default_rng(42)
sample_size = min(20_000, len(y))
idx = rng.choice(len(y), sample_size, replace=False)
X_sample = X_scaled[idx]
y_sample = y[idx]

if umap is not None:
    # TODO: Build umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
    reducer = ____
    # TODO: Fit-transform X_sample
    embedding_umap = ____
    print(f"\nUMAP embedding: {embedding_umap.shape}")
else:
    from sklearn.decomposition import PCA

    embedding_umap = PCA(n_components=2, random_state=42).fit_transform(X_sample)
    print("\nUMAP not installed, using PCA fallback")



## TASK 2: Compare with t-SNE


In [ ]:
from sklearn.manifold import TSNE

# TODO: Build TSNE(n_components=2, perplexity=30, random_state=42)
tsne = ____
# TODO: fit_transform on X_sample[:5000] (t-SNE is O(n²))
embedding_tsne = ____

print(f"t-SNE embedding: {embedding_tsne.shape}")
print("\nUMAP vs t-SNE:")
print("  UMAP: preserves global structure, faster, supports transform()")
print("  t-SNE: better local structure, O(n²), no out-of-sample transform")



## TASK 3: Isolation Forest


In [ ]:
# Theory: anomalies are isolated in fewer random partitions.

# TODO: Build IsolationForest(n_estimators=200, contamination=0.002,
#       random_state=42, n_jobs=-1)
iso_forest = ____
# TODO: Fit on X_scaled and compute anomaly scores
# Hint: -iso_forest.fit(X_scaled).score_samples(X_scaled) — higher = more anomalous
iso_scores = ____
# TODO: Predict labels (-1 = anomaly, 1 = normal)
iso_labels = ____  # Hint: iso_forest.predict(X_scaled)

# TODO: Compute ROC-AUC and average precision against y
iso_auc = ____  # Hint: roc_auc_score(y, iso_scores)
iso_ap = ____  # Hint: average_precision_score(y, iso_scores)

print(f"\n=== Isolation Forest ===")
print(f"AUC-ROC: {iso_auc:.4f}")
print(f"Average Precision: {iso_ap:.4f}")
print(f"Predicted anomalies: {(iso_labels == -1).sum():,} / {len(iso_labels):,}")
print(f"True frauds: {y.sum():.0f}")



## TASK 4: Local Outlier Factor


In [ ]:
# TODO: Build LocalOutlierFactor(n_neighbors=20, contamination=0.002, novelty=False)
lof = ____
# TODO: fit_predict on X_scaled
lof_labels = ____
# TODO: Negate the negative_outlier_factor_ so higher = more anomalous
lof_scores = ____  # Hint: -lof.negative_outlier_factor_

# TODO: Compute ROC-AUC and average precision
lof_auc = ____
lof_ap = ____

print(f"\n=== Local Outlier Factor ===")
print(f"AUC-ROC: {lof_auc:.4f}")
print(f"Average Precision: {lof_ap:.4f}")



## TASK 5: Ensemble anomaly scores with EnsembleEngine.blend()


In [ ]:
def normalise_scores(scores: np.ndarray) -> np.ndarray:
    return (scores - scores.min()) / (scores.max() - scores.min() + 1e-10)


iso_norm = normalise_scores(iso_scores)
lof_norm = normalise_scores(lof_scores)

# Method A: AUC-weighted average
w_iso = iso_auc / (iso_auc + lof_auc)
w_lof = lof_auc / (iso_auc + lof_auc)
ensemble_scores_manual = w_iso * iso_norm + w_lof * lof_norm

# Method B: EnsembleEngine.blend() with sklearn-compatible wrappers
from kailash_ml import EnsembleEngine
from sklearn.base import BaseEstimator, ClassifierMixin


class AnomalyScorer(BaseEstimator, ClassifierMixin):
    """Wraps an anomaly detector to expose predict_proba for EnsembleEngine."""

    def __init__(self, detector=None, scores=None):
        self.detector = detector
        self.scores = scores

    def __sklearn_tags__(self):
        from sklearn.utils._tags import ClassifierTags

        tags = super().__sklearn_tags__()
        tags.estimator_type = "classifier"
        tags.classifier_tags = ClassifierTags()
        return tags

    def fit(self, X, y=None):
        self.classes_ = np.array([0, 1])
        return self

    def predict_proba(self, X):
        norm = normalise_scores(self.scores[: len(X)])
        return np.column_stack([1 - norm, norm])

    def predict(self, X):
        return (self.scores[: len(X)] > np.median(self.scores)).astype(int)


iso_scorer = AnomalyScorer(iso_forest, iso_scores)
lof_scorer = AnomalyScorer(lof, lof_scores)

# TODO: Instantiate the EnsembleEngine
engine = ____
blend_df = pl.DataFrame(
    {f"f{i}": X_scaled[:, i] for i in range(X_scaled.shape[1])}
).with_columns(pl.Series("is_fraud", y))

# TODO: Call engine.blend with both scorers, target="is_fraud", weights=[w_iso, w_lof],
#       method="soft"
blend_result = ____

blend_features = blend_df.drop("is_fraud")
X_blend = blend_features.to_numpy()
ensemble_scores = blend_result.ensemble_model.predict_proba(X_blend)[:, 1]

# TODO: Compute ensemble ROC-AUC and average precision
ensemble_auc = ____
ensemble_ap = ____

print(f"\n=== Ensemble via EnsembleEngine.blend() ===")
print(f"Weights: IF={w_iso:.3f}, LOF={w_lof:.3f}")
print(f"AUC-ROC: {ensemble_auc:.4f}")
print(f"Average Precision: {ensemble_ap:.4f}")

print(f"\nImprovement over best individual:")
best_individual = max(iso_auc, lof_auc)
print(f"  AUC-ROC: {ensemble_auc - best_individual:+.4f}")

print("\nEnsembleEngine methods:")
print("  blend()  — weighted average of predictions (soft voting)")
print("  stack()  — meta-learner trained on base model outputs")
print("  bag()    — bootstrap aggregation (bagging)")
print("  boost()  — sequential boosting on residuals")



## TASK 6: Evaluate and visualise


In [ ]:
viz = ModelVisualizer()

for name, scores in [
    ("IsolationForest", iso_scores),
    ("LOF", lof_scores),
    ("Ensemble", ensemble_scores),
]:
    fig = viz.roc_curve(y, scores)
    fig.update_layout(title=f"ROC: {name}")
    fig.write_html(f"ex2_roc_{name.lower()}.html")

# TODO: Build the comparison dict for metric_comparison
# Hint: keys = "Isolation Forest", "LOF", "Ensemble"; values = {"AUC_ROC": ..., "Avg_Precision": ...}
comparison = ____
fig = viz.metric_comparison(comparison)
fig.update_layout(title="Anomaly Detection Comparison")
fig.write_html("ex2_anomaly_comparison.html")
print("\nSaved: ex2_anomaly_comparison.html")

precision, recall, thresholds = precision_recall_curve(y, ensemble_scores)
idx_80 = min(np.searchsorted(-recall[::-1], -0.80), len(precision) - 1)
print(f"\nAt recall=0.80: precision={precision[idx_80]:.4f}")

print("\n✓ Exercise 4 complete — UMAP + anomaly detection ensemble")

